# 2.2 — Diseno de productos de datos

## Unidad 2: Caracteristicas de Productos de Datos

Un producto de datos que solo funciona en tu maquina, que solo tu entiendes y que se rompe cuando algo cambia no es un producto — es un script personal. Un producto de datos real es compartible, modular, documentado y versionado. Otro equipo puede usarlo sin preguntarte como funciona.

Este notebook cubre los principios y herramientas para disenar productos de datos que sobrevivan a su creador.

### Contenido:
1. Principios de diseno
2. Contratos de datos
3. Modularidad y reutilizacion
4. Versionado de datos
5. Metadata y documentacion
6. Formatos interoperables

In [ ]:
import pandas as pd
import numpy as np
import json
import os
from datetime import datetime

---
## 1. Principios de diseno

Antes de las herramientas, los principios. Un producto de datos bien disenado cumple estas caracteristicas:

| Principio | Pregunta clave | Ejemplo malo | Ejemplo bueno |
|---|---|---|---|
| **Compartible** | Puede usarlo alguien que no lo construyo? | Script que solo corre en tu maquina | Paquete instalable con instrucciones |
| **Interoperable** | Puede conectarse con otros sistemas? | CSV con encoding raro y fechas ambiguas | Parquet con tipos explicitos y schema |
| **Modular** | Puedo cambiar una parte sin romper todo? | Un notebook de 2000 lineas | Funciones separadas por responsabilidad |
| **Reutilizable** | Puedo usarlo para otro proyecto? | Codigo con rutas hardcoded y variables globales | Funciones parametrizadas, config externa |
| **Documentado** | Alguien puede entenderlo sin preguntarme? | Funciones sin docstring, columnas sin descripcion | README, docstrings, catalogo de datos |
| **Versionado** | Puedo saber que cambio y cuando? | Archivos con nombre "final_v3_definitivo" | Git + versionado semantico |

---
## 2. Contratos de datos

Un contrato de datos es un acuerdo entre quien produce los datos y quien los consume. Define que columnas tiene, que tipo, que valores acepta y que garantias ofrece. Si el productor cambia la estructura sin avisar, el contrato se rompe y el consumidor lo detecta automaticamente.

Es la diferencia entre "me mandaron un CSV y tuve que adivinar que significaba cada columna" y "hay un documento que me dice exactamente que esperar".

In [ ]:
# ============================================================
# CONTRATO DE DATOS COMO ARCHIVO JSON
# ============================================================

# Un contrato define la estructura esperada de un dataset
# Se guarda como archivo y se versiona con el proyecto

contrato_ventas = {
    "nombre": "transacciones_ventas",
    "version": "1.0.0",
    "descripcion": "Registro de transacciones de venta de productos de datos",
    "propietario": "equipo-datos",
    "consumidores": ["dashboard-ventas", "reporte-gerencia", "modelo-prediccion"],
    "frecuencia_actualizacion": "diaria",
    "sla": {
        "disponibilidad": "99.5%",
        "latencia_maxima": "2 horas despues del cierre del dia",
        "contacto": "datos@empresa.com"
    },
    "columnas": {
        "fecha": {
            "tipo": "datetime",
            "nullable": False,
            "descripcion": "Fecha de la transaccion",
            "formato": "YYYY-MM-DD"
        },
        "producto": {
            "tipo": "string",
            "nullable": False,
            "valores_permitidos": ["dashboard", "reporte", "api", "app web", "pipeline etl"],
            "descripcion": "Tipo de producto vendido"
        },
        "region": {
            "tipo": "string",
            "nullable": False,
            "valores_permitidos": ["bogota", "medellin", "cali", "manizales", "barranquilla"],
            "descripcion": "Ciudad de la venta"
        },
        "unidades": {
            "tipo": "integer",
            "nullable": False,
            "rango": {"min": 1, "max": 200},
            "descripcion": "Cantidad de unidades vendidas"
        },
        "precio": {
            "tipo": "float",
            "nullable": False,
            "rango": {"min": 0.01},
            "unidad": "COP",
            "descripcion": "Precio unitario en pesos colombianos"
        },
        "calificacion": {
            "tipo": "integer",
            "nullable": True,
            "rango": {"min": 1, "max": 5},
            "descripcion": "Calificacion del cliente (1=peor, 5=mejor)"
        }
    }
}

# Guardar el contrato
with open("contrato_ventas.json", "w", encoding="utf-8") as f:
    json.dump(contrato_ventas, f, indent=2, ensure_ascii=False)

print("Contrato guardado: contrato_ventas.json")
print(f"Columnas definidas: {list(contrato_ventas['columnas'].keys())}")
print(f"Consumidores: {contrato_ventas['consumidores']}")

In [ ]:
# ============================================================
# GENERAR ESQUEMA PANDERA DESDE EL CONTRATO
# ============================================================

# El contrato es documentacion. El esquema es ejecucion.
# Esta funcion convierte uno en otro automaticamente.

import pandera as pa

def contrato_a_esquema(contrato):
    """
    Convierte un contrato de datos JSON en un esquema pandera.
    El contrato define las reglas, el esquema las ejecuta.
    """
    columnas = {}
    
    tipo_map = {
        "string": str,
        "integer": int,
        "float": float,
        "datetime": "datetime64[ns]",
        "boolean": bool,
    }
    
    for col_name, col_def in contrato["columnas"].items():
        checks = []
        
        # Valores permitidos
        if "valores_permitidos" in col_def:
            checks.append(pa.Check.isin(col_def["valores_permitidos"]))
        
        # Rango
        if "rango" in col_def:
            rango = col_def["rango"]
            if "min" in rango and "max" in rango:
                checks.append(pa.Check.in_range(rango["min"], rango["max"]))
            elif "min" in rango:
                checks.append(pa.Check.greater_than_or_equal_to(rango["min"]))
            elif "max" in rango:
                checks.append(pa.Check.less_than_or_equal_to(rango["max"]))
        
        tipo = tipo_map.get(col_def["tipo"], str)
        nullable = col_def.get("nullable", True)
        
        columnas[col_name] = pa.Column(tipo, checks=checks, nullable=nullable)
    
    return pa.DataFrameSchema(columnas)

# Generar esquema desde el contrato
esquema = contrato_a_esquema(contrato_ventas)
print("Esquema pandera generado desde el contrato.")
print(f"Columnas: {list(esquema.columns.keys())}")

In [ ]:
# ============================================================
# VERIFICAR QUE EL CONTRATO SE CUMPLE
# ============================================================

datos = pd.DataFrame({
    "fecha": pd.to_datetime(["2024-06-01", "2024-06-02", "2024-06-03"]),
    "producto": ["dashboard", "reporte", "api"],
    "region": ["bogota", "medellin", "cali"],
    "unidades": [15, 8, 22],
    "precio": [450.0, 280.0, 620.0],
    "calificacion": [4, 5, 3],
})

try:
    esquema.validate(datos)
    print("El dataset cumple el contrato.")
except pa.errors.SchemaError as e:
    print(f"El dataset rompe el contrato: {e}")

---
## 3. Modularidad y reutilizacion

Un notebook de 2000 lineas no es reutilizable. Una funcion con parametros si. Un modulo con funciones agrupadas por responsabilidad es todavia mejor. La idea es separar el codigo en piezas que se pueden usar independientemente.

In [ ]:
# ============================================================
# MAL: todo en un bloque, hardcoded, no reutilizable
# ============================================================

# Esto funciona pero no se puede reutilizar en otro proyecto

# df = pd.read_csv("/home/sergio/datos/ventas_junio.csv")
# df = df[df["region"] == "bogota"]
# df["ingreso"] = df["unidades"] * df["precio"]
# resumen = df.groupby("producto")["ingreso"].sum()
# resumen.to_csv("/home/sergio/reportes/resumen_bogota.csv")

print("Problemas:")
print("  - Rutas hardcoded (solo funciona en tu maquina)")
print("  - Region fija (no sirve para medellin)")
print("  - Sin validacion (si el CSV cambia, se rompe en silencio)")
print("  - Sin documentacion (que hace cada linea?)")

In [ ]:
# ============================================================
# BIEN: funciones separadas por responsabilidad
# ============================================================

def cargar_ventas(ruta):
    """Carga el CSV de ventas y convierte tipos."""
    df = pd.read_csv(ruta)
    df["fecha"] = pd.to_datetime(df["fecha"])
    return df

def filtrar_por_region(df, region):
    """Filtra un DataFrame de ventas por region."""
    return df[df["region"] == region].copy()

def calcular_ingreso(df):
    """Agrega la columna ingreso = unidades * precio."""
    df = df.copy()
    df["ingreso"] = df["unidades"] * df["precio"]
    return df

def resumir_por_producto(df):
    """Genera resumen de ingreso agrupado por producto."""
    return (df.groupby("producto")["ingreso"]
            .agg(["sum", "mean", "count"])
            .rename(columns={"sum": "total", "mean": "promedio", "count": "ventas"})
            .sort_values("total", ascending=False))

# Ahora el pipeline es legible y reutilizable
# df = cargar_ventas("ventas.csv")
# df = filtrar_por_region(df, "bogota")  # parametro, no hardcoded
# df = calcular_ingreso(df)
# resumen = resumir_por_producto(df)

print("Ventajas:")
print("  - Cada funcion hace UNA cosa")
print("  - Se pueden reusar en otro proyecto")
print("  - Se pueden testear individualmente")
print("  - El pipeline se lee como un texto")

In [ ]:
# ============================================================
# MEJOR: modulo .py con configuracion externa
# ============================================================

# En un proyecto real, las funciones van en un archivo .py
# y la configuracion en un archivo separado

# Estructura recomendada:
estructura = """
mi_producto/
    config.yaml           # Configuracion (rutas, parametros)
    contrato_ventas.json  # Contrato de datos
    
    src/
        __init__.py
        cargar.py         # Funciones de carga
        limpiar.py        # Funciones de limpieza
        transformar.py    # Funciones de transformacion
        validar.py        # Esquemas de validacion
    
    notebooks/
        exploracion.ipynb  # Analisis interactivo
    
    tests/
        test_limpiar.py    # Pruebas unitarias
    
    README.md              # Documentacion
    requirements.txt       # Dependencias
"""

print(estructura)

In [ ]:
# ============================================================
# CONFIGURACION EXTERNA CON YAML
# ============================================================

# La configuracion no va en el codigo — va en un archivo aparte
# Asi el mismo codigo funciona con datos distintos

config_yaml = """
# config.yaml
datos:
  ruta_entrada: "datos/ventas_crudas.csv"
  ruta_salida: "datos/ventas_limpias.parquet"
  contrato: "contrato_ventas.json"

procesamiento:
  regiones_validas:
    - bogota
    - medellin
    - cali
    - manizales
    - barranquilla
  precio_maximo: 10000
  unidades_maximo: 200

salida:
  formato: "parquet"
  compresion: "snappy"
"""

with open("config.yaml", "w") as f:
    f.write(config_yaml)

print("config.yaml creado.")
print("El codigo lee la config, no tiene valores hardcoded.")
print("Para cambiar la ruta o un parametro, editas el YAML, no el codigo.")

In [ ]:
# ============================================================
# LEER CONFIGURACION EN PYTHON
# ============================================================

# pip install pyyaml
import yaml

with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

print("Configuracion cargada:")
print(f"  Ruta entrada: {config['datos']['ruta_entrada']}")
print(f"  Formato salida: {config['salida']['formato']}")
print(f"  Regiones: {config['procesamiento']['regiones_validas']}")

# Ahora el codigo usa config['datos']['ruta_entrada'] en vez de un string hardcoded

---
## 4. Versionado de datos

El codigo se versiona con Git. Pero los datos? Si el esquema cambia, si se agrega una columna, si se corrige un error historico, necesitamos saber que version de los datos uso cada reporte.

In [ ]:
# ============================================================
# VERSIONADO SIMPLE — POR NOMBRE DE ARCHIVO
# ============================================================

# La forma mas basica: incluir fecha o version en el nombre

# MAL:
# ventas.csv
# ventas_final.csv
# ventas_final_v2.csv
# ventas_final_v2_definitivo.csv

# BIEN:
# ventas_2024-06-01.parquet
# ventas_2024-06-02.parquet

# MEJOR: con version semantica del esquema
# ventas_v1.0.0_2024-06-01.parquet
# ventas_v1.1.0_2024-06-15.parquet  (se agrego una columna)

def guardar_con_version(df, nombre, version, carpeta="datos_versionados"):
    """Guarda un DataFrame con version y timestamp en el nombre."""
    os.makedirs(carpeta, exist_ok=True)
    fecha = datetime.now().strftime("%Y%m%d")
    ruta = os.path.join(carpeta, f"{nombre}_v{version}_{fecha}.parquet")
    df.to_parquet(ruta)
    print(f"Guardado: {ruta}")
    return ruta

guardar_con_version(datos, "ventas", "1.0.0")

In [ ]:
# ============================================================
# VERSIONADO SEMANTICO PARA DATOS
# ============================================================

# El mismo concepto que en software, aplicado a datos:
#
# MAJOR.MINOR.PATCH
#   1.0.0 -> 1.0.1   Correccion de datos (arreglar un error historico)
#   1.0.0 -> 1.1.0   Columna nueva (compatible hacia atras)
#   1.0.0 -> 2.0.0   Cambio de esquema (rompe compatibilidad)

historial_versiones = [
    {"version": "1.0.0", "fecha": "2024-01-15",
     "cambios": "Version inicial. 6 columnas."},
    
    {"version": "1.0.1", "fecha": "2024-02-01",
     "cambios": "Corregido: precios negativos en enero reemplazados con mediana."},
    
    {"version": "1.1.0", "fecha": "2024-03-10",
     "cambios": "Nueva columna: vendedor. Compatible hacia atras."},
    
    {"version": "2.0.0", "fecha": "2024-06-01",
     "cambios": "Columna 'precio' renombrada a 'precio_unitario'. Rompe compatibilidad."},
]

print("Historial de versiones:")
for v in historial_versiones:
    print(f"  v{v['version']} ({v['fecha']}): {v['cambios']}")

In [ ]:
# ============================================================
# CHANGELOG COMO ARCHIVO
# ============================================================

changelog = """# Changelog — transacciones_ventas

## [2.0.0] - 2024-06-01
### Breaking
- Columna 'precio' renombrada a 'precio_unitario'
- Consumidores deben actualizar sus queries

## [1.1.0] - 2024-03-10
### Added
- Nueva columna 'vendedor' (nullable)
- Compatible hacia atras, los consumidores existentes no se rompen

## [1.0.1] - 2024-02-01
### Fixed
- Corregidos 23 precios negativos de enero (reemplazados con mediana del producto)

## [1.0.0] - 2024-01-15
### Initial
- Version inicial: fecha, producto, region, unidades, precio, calificacion
"""

with open("CHANGELOG.md", "w") as f:
    f.write(changelog)

print("CHANGELOG.md creado.")
print("Este archivo se versiona con Git junto al contrato y el codigo.")

---
## 5. Metadata y documentacion

La metadata es informacion sobre los datos: que significan, de donde vienen, cuando se actualizaron, que tan confiables son. Sin metadata, un archivo es solo columnas con nombres crípticos.

In [ ]:
# ============================================================
# CATALOGO DE DATOS — diccionario de columnas
# ============================================================

# Un catalogo de datos describe cada columna para que
# cualquier persona pueda entender el dataset sin preguntar

catalogo = pd.DataFrame([
    {"columna": "fecha", "tipo": "datetime",
     "descripcion": "Fecha en que se realizo la transaccion",
     "fuente": "Sistema ERP", "ejemplo": "2024-06-15",
     "notas": "Zona horaria: America/Bogota"},
    
    {"columna": "producto", "tipo": "string",
     "descripcion": "Tipo de producto de datos vendido",
     "fuente": "Catalogo de productos", "ejemplo": "dashboard",
     "notas": "5 valores validos, siempre en minusculas"},
    
    {"columna": "region", "tipo": "string",
     "descripcion": "Ciudad donde se realizo la venta",
     "fuente": "CRM", "ejemplo": "bogota",
     "notas": "Sin tildes, en minusculas"},
    
    {"columna": "unidades", "tipo": "integer",
     "descripcion": "Cantidad de licencias o suscripciones vendidas",
     "fuente": "Sistema ERP", "ejemplo": "15",
     "notas": "Rango valido: 1-200. Valores mayores son errores de captura"},
    
    {"columna": "precio", "tipo": "float",
     "descripcion": "Precio por unidad en pesos colombianos (COP)",
     "fuente": "Lista de precios", "ejemplo": "450.00",
     "notas": "No incluye IVA. Siempre positivo"},
    
    {"columna": "calificacion", "tipo": "integer",
     "descripcion": "Calificacion del cliente despues de la compra",
     "fuente": "Encuesta post-venta", "ejemplo": "4",
     "notas": "Escala 1-5. Puede ser nulo si el cliente no respondio"},
])

print("Catalogo de datos:")
catalogo

In [ ]:
# ============================================================
# METADATA AUTOMATICA — informacion del dataset
# ============================================================

def generar_metadata(df, nombre, version, descripcion):
    """
    Genera metadata automatica de un DataFrame.
    Incluye estadisticas, tipos y timestamp.
    """
    meta = {
        "nombre": nombre,
        "version": version,
        "descripcion": descripcion,
        "generado_el": datetime.now().isoformat(),
        "filas": len(df),
        "columnas": len(df.columns),
        "tamano_bytes": df.memory_usage(deep=True).sum(),
        "nulos_totales": int(df.isnull().sum().sum()),
        "duplicados": int(df.duplicated().sum()),
        "esquema": {
            col: {
                "tipo": str(df[col].dtype),
                "nulos": int(df[col].isnull().sum()),
                "unicos": int(df[col].nunique()),
            }
            for col in df.columns
        },
        "periodo": {},
    }
    
    # Si hay columna de fecha, agregar rango temporal
    for col in df.select_dtypes(include=["datetime64"]).columns:
        meta["periodo"] = {
            "inicio": df[col].min().isoformat(),
            "fin": df[col].max().isoformat(),
        }
    
    return meta

meta = generar_metadata(datos, "transacciones_ventas", "1.0.0",
                        "Ventas de productos de datos")

# Guardar metadata junto al dataset
with open("ventas_metadata.json", "w") as f:
    json.dump(meta, f, indent=2, default=str)

print("Metadata generada:")
print(json.dumps(meta, indent=2, default=str))

In [ ]:
# ============================================================
# README PARA EL PRODUCTO DE DATOS
# ============================================================

readme = """# Transacciones de Ventas

## Que es
Dataset diario con las transacciones de venta de productos de datos.

## Propietario
Equipo de Datos (datos@empresa.com)

## Actualizacion
Diaria, antes de las 8:00 AM.

## Esquema
Ver `contrato_ventas.json` para la definicion completa.

| Columna | Tipo | Descripcion |
|---|---|---|
| fecha | datetime | Fecha de la transaccion |
| producto | string | Tipo de producto (5 valores) |
| region | string | Ciudad (5 valores) |
| unidades | int | Cantidad vendida (1-200) |
| precio | float | Precio unitario COP |
| calificacion | int | Nota del cliente (1-5, nullable) |

## Como usar
```python
import pandas as pd
df = pd.read_parquet("datos/ventas_v1.0.0_20240601.parquet")
```

## Cambios
Ver `CHANGELOG.md`.
"""

with open("README_datos.md", "w") as f:
    f.write(readme)

print("README_datos.md creado.")
print("Con esto, cualquier persona del equipo sabe que hay, como usarlo y a quien preguntar.")

---
## 6. Formatos interoperables

No todos los consumidores usan Python. Un dashboard puede estar en Power BI, un modelo en R, una API en Go. El formato del archivo determina quien puede leerlo y que tan eficiente es.

In [ ]:
# ============================================================
# COMPARAR FORMATOS — tamano y velocidad
# ============================================================

import time

# Crear un dataset mas grande para que la diferencia sea visible
np.random.seed(42)
n = 100_000
df_grande = pd.DataFrame({
    "fecha": pd.date_range("2023-01-01", periods=n, freq="h"),
    "producto": np.random.choice(["dashboard", "reporte", "api", "app web"], n),
    "region": np.random.choice(["bogota", "medellin", "cali", "manizales"], n),
    "unidades": np.random.randint(1, 50, n),
    "precio": np.round(np.random.uniform(100, 800, n), 2),
})

resultados = []

# CSV
t0 = time.time()
df_grande.to_csv("test.csv", index=False)
t_write_csv = time.time() - t0
size_csv = os.path.getsize("test.csv")

t0 = time.time()
pd.read_csv("test.csv")
t_read_csv = time.time() - t0

resultados.append({"formato": "CSV", "tamano_mb": round(size_csv / 1e6, 2),
                   "escribir_seg": round(t_write_csv, 3), "leer_seg": round(t_read_csv, 3),
                   "tipos_preservados": "No", "compresion": "No"})

# Parquet
t0 = time.time()
df_grande.to_parquet("test.parquet")
t_write_pq = time.time() - t0
size_pq = os.path.getsize("test.parquet")

t0 = time.time()
pd.read_parquet("test.parquet")
t_read_pq = time.time() - t0

resultados.append({"formato": "Parquet", "tamano_mb": round(size_pq / 1e6, 2),
                   "escribir_seg": round(t_write_pq, 3), "leer_seg": round(t_read_pq, 3),
                   "tipos_preservados": "Si", "compresion": "Si (snappy)"})

# JSON
t0 = time.time()
df_grande.to_json("test.json", orient="records")
t_write_json = time.time() - t0
size_json = os.path.getsize("test.json")

t0 = time.time()
pd.read_json("test.json")
t_read_json = time.time() - t0

resultados.append({"formato": "JSON", "tamano_mb": round(size_json / 1e6, 2),
                   "escribir_seg": round(t_write_json, 3), "leer_seg": round(t_read_json, 3),
                   "tipos_preservados": "Parcial", "compresion": "No"})

# Excel
t0 = time.time()
df_grande.head(50000).to_excel("test.xlsx", index=False)  # Excel tiene limite de filas
t_write_xl = time.time() - t0
size_xl = os.path.getsize("test.xlsx")

resultados.append({"formato": "Excel", "tamano_mb": round(size_xl / 1e6, 2),
                   "escribir_seg": round(t_write_xl, 3), "leer_seg": "-",
                   "tipos_preservados": "Parcial", "compresion": "Si (zip)"})

df_comparacion = pd.DataFrame(resultados)
print(f"Benchmark con {n:,} filas:")
print()
df_comparacion

In [ ]:
# ============================================================
# POR QUE PARQUET
# ============================================================

# Parquet es el formato recomendado para productos de datos:

ventajas = [
    "Columnar: lee solo las columnas que necesitas, no todo el archivo",
    "Tipos preservados: datetime sigue siendo datetime, no string",
    "Compresion integrada: ocupa menos espacio que CSV",
    "Rapido: lectura y escritura mas rapida que CSV",
    "Compatible: Python, R, Spark, Power BI, DuckDB lo leen",
    "Metadata integrada: el esquema viaja dentro del archivo",
]

print("Ventajas de Parquet:")
for v in ventajas:
    print(f"  - {v}")

print("\nCuando NO usar Parquet:")
print("  - Cuando el consumidor necesita abrir el archivo en Excel")
print("  - Cuando es un archivo pequeno que va por correo")
print("  - En esos casos, CSV o Excel siguen siendo validos")

In [ ]:
# Limpiar archivos de prueba
for f in ["test.csv", "test.parquet", "test.json", "test.xlsx"]:
    if os.path.exists(f):
        os.remove(f)

---
## Resumen

| Concepto | Lo que importa |
|---|---|
| **Contrato de datos** | Acuerdo entre productor y consumidor: que columnas, que tipos, que garantias |
| **Modularidad** | Funciones separadas por responsabilidad, config externa, estructura de proyecto |
| **Versionado** | Semantico (MAJOR.MINOR.PATCH), changelog, historial de cambios |
| **Metadata** | Catalogo de columnas, estadisticas automaticas, README |
| **Interoperabilidad** | Parquet para produccion, CSV para compartir con no-tecnicos |

### La regla de oro

Si te vas de vacaciones y alguien mas tiene que usar tu producto de datos sin llamarte, esta bien disenado. Si te tienen que llamar, falta documentacion, falta un contrato o falta modularidad.

### Siguiente paso
En el **Notebook 2.3** definiremos KPIs y metricas de producto: que medir, por que, y como saber si el producto de datos esta generando valor.